# How to plug the memory into the chains
1. LLM Chain
    - off-the-shelf(=general purpose) chain
2. Custom Chain
    - LangChain Expression Language를 활용하여 직접 만든 체인

In [ ]:
'''
# 5.5 Memory on LLM Chain

from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=80,
    # 컨텐츠를 template {chat_history} 변수 안에 넣어라
    memory_key="chat_history",
)

template = """
    You are a helpful AI talking to a human.

    {chat_history}
    Human: {question}
    You:
"""

chain = LLMChain(
    llm=llm,
    memory=memory,
    # 아래와 같이만 작성하면 메모리는 업데이트 중이나 verbose로 확인해보면 프롬프트에 대화 내역이 추가되고 있지 않다. 따라서 template 작성.
    # prompt=PromptTemplate.from_template("{question}"),
    verbose=True,
    prompt=PromptTemplate.from_template(template),
)
'''

# 5.6 Chat-based memory
- memory.load_memory_variables() 실행시, 메모리 클래스는 프롬프트에 메모리를 두 가지 방식으로 출력할 수 있다.
> 1. String: 기본값
> 2. Message: 대화 기반으로 사용하고 싶다면 설정을 아래와 같이 바꿔야 한다.

In [ ]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history",
    # Message Class로 변경
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a english tutor who helps kids with speaking english."),
    # human message인지, system message인지, AI message인지, 그리고 메시지 양을 예측하기 어려울 때 제한 없이 넣을 수 있다.
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=prompt,
    verbose=True,
)

In [ ]:
chain.predict(question="My name is Nico. I live in Seoul")
chain.predict(question="What is my name?")

memory.load_memory_variables({})

In [ ]:
# adding a memory to a Custom Chain.

from langchain.chat_models import openai
# prompt가 format 되기 전에 함수를 실행시킨다.
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationSummaryBufferMemory


llm = ChatOpenAI(temperature=0.1)

# memory and prompt doesn't change from base LLM Chain.
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI that talking to a human"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)

# 메모리 변수를 획득하는 함수 / _는 에러 무시
def load_memory(_):
    memory.load_memory_variables({})["history"]

# 체인을 호출하는 메모리를 수동으로 관리하는 함수
def invoke_chain(question):
    result = chain.invoke(
        {"question": question},
        )
    memory.save_context(
        {"input": question},
        {"output": result.content},
    )

'''load_memory 함수 먼저 실행하여 그 결과를 prompt가 필요로 하는 history의 input으로 넣어준다.
즉, 아래와 같다.
chain.invoke({
    "history": load_memory(),
    "question": "I'm a Pink Pong",
})
'''
chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm

invoke_chain("My name is Nico")
invoke_chain("What is your name")